# JAX on TIKE
When we're running computationally expensive code on the cloud, there are three considerations we might have:

1. We want to <span style="color:green">*optimize*</span> our code
2. We want to <span style="color:green">*parallelize*</span> our code across multiple cores
3. We want access to <span style="color:green">*gradients*</span> of our functions to accelerate our algorithms

[JAX](https://docs.jax.dev/en/latest/automatic-differentiation.html) is a Python library that helps with all three goals. In this and the following notebooks, we will briefly introduce ``JAX`` and how to effectively use it on TIKE.

First, let's install ``JAX`` in this kernel.

In [2]:
!pip install -U jax

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 9.9 MB/s eta 0:00:00ta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.1/81.1 MB 86.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 20.0 MB/s eta 0:00:00ta 0:00:01
  Attempting uninstall: ml_dtypes
    Found existing installation: ml-dtypes 0.3.2
    Uninstalling ml-dtypes-0.3.2:
      Successfully uninstalled ml-dtypes-0.3.2
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [jax]2/3 [jax]ib]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow 2.16.1 requires ml-dtypes~=0.3.1, but you have ml-dtypes 0.5.3 which is incompatible.


Next, let's import the package. We'll also import `numpy` and the `time` module for comparison.

In [71]:
import jax.numpy as jnp
import numpy as np
import time

from jax import grad

## Arrays in `JAX`

Let's start by reviewing some array operations in `numpy`. We can create arrays of values like so:

In [9]:
array1 = np.array([1,2,3])
array2 = np.array([2,4,5])

These arrays obey some standard operations, such as addition...

In [11]:
array3 = array1 + array2
array3

array([3, 6, 8])

...multiplication...

In [12]:
array4 = array1 * array2
array4

array([ 2,  8, 15])

...and indexing.

In [13]:
val1 = array1[0]
val1

1

We can also iterate through arrays to sequentially access the values within them:

In [14]:
for value in array1:
    print(value)

1
2
3


`JAX` provides objects that are very similar to `numpy` arrays. This is by design, so that there is maximal interoperability between `numpy`-based code and `JAX`-based code. 

Let's create some `JAX` arrays!

In [16]:
jax_array1 = jnp.array([1,2,3])
jax_array2 = jnp.array([2,4,5])

You can perform many of the same operations on `JAX` arrays as `numpy` arrays:

In [18]:
jax_array3 = jax_array1 + jax_array2
jax_array3

Array([3, 6, 8], dtype=int32)

In [19]:
jax_array4 = jax_array1 * jax_array2
jax_array4

Array([ 2,  8, 15], dtype=int32)

There are some important caveats, though! One of the biggest logical shifts between writing `JAX` code and `numpy` code is that `JAX` arrays are immutable. Let's see what that looks like.

In [20]:
array1[0] = 100 # changing a numpy array value, no problem
array1

array([100,   2,   3])

In [21]:
jax_array1[0] = 100 # changing a JAX array value, and...
jax_array1

TypeError: JAX arrays are immutable and do not support in-place item assignment. Instead of x[idx] = y, use x = x.at[idx].set(y) or another .at[] method: https://docs.jax.dev/en/latest/_autosummary/jax.numpy.ndarray.at.html

This seems quite inconvenient! As the error above shows, we have to use more verbose syntax to change the values inside a `JAX` array. And as it turns out, we aren't actually changing that same array even when we use that syntax — we're creating a modified copy of the original array.

Why would we accept this seemingly more restricted version of `numpy`? As we'll see, design choices such as this one allow `JAX` to be more efficient down the road.

## Functions, `JAX`, and JIT
One of the easiest ways to see the benefits of `JAX` is by speeding up a single function. Let's begin with a simple function implemented in `numpy`:

In [27]:
def numpy_function(x):
    """
    A simple numpy-based function that invokes some trigonometric functions.

    Inputs
    ------
        :x: (array) input array, of floats.

    Outputs
    -------
        :result: (array) result of applying trigonometric functions. Should be equal to 1.
    """
    return np.sin(x) ** 2 + np.cos(x) ** 2

Let's create the same function in `JAX`. Again, the only change we'll have to make in our syntax is swapping `np` for `jnp`!

In [38]:
def jax_function(x):
    """
    A simple JAX-based function that invokes some trigonometric functions.

    Inputs
    ------
        :x: (array) input array, of floats.

    Outputs
    -------
        :result: (array) result of applying trigonometric functions. Should be equal to 1.
    """
    return jnp.sin(x) ** 2 + jnp.cos(x) ** 2

Now, let's define large input arrays. `JAX` tends to perform comparatively better over a large number of calculations. We'll create this in both `JAX` and `numpy` arrays.

In [39]:
x_np = np.linspace(0, 100, 10_000_000)
x_jax = jnp.linspace(0, 100, 10_000_000)

Let's see how long it takes to execute the `numpy` function. We can use the ``%%timeit`` magic to time this code over multiple evaluations. Doing so gives us a representative value for how fast the code is.

In [72]:
%%timeit
y_np = numpy_function(x_np)
t1 = time.time()

311 ms ± 25.8 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


Neat! And now for the `JAX` function ...

In [80]:
%%timeit
y_jax = jax_function(x_jax)

130 ms ± 14.2 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


Wow! This is already much faster (between 2x and 3x speedup).

We can further accelerate things by leveraging `JAX`'s "just-in-time", or JIT, compilation. By using the [OpenXLA backend](https://openxla.org/xla), `JAX` will read through your Python function on the first JIT execution, keeping track of Python operations as it goes. It then "fuses" the operations together into more efficient, vectorized machine code. This first pass-through can be a bit slower than `numpy`, because the program has to keep track of the speed gains it'd like to make. On subsequent runs, though, the vectorized function is remembered ("cached"), so the compilation cost is bypassed!

Let's take a look at this in practice.

In [77]:
f_jax_jit = jax.jit(jax_function)

With the "jitted" function now available, let's time it. First, we'll use the `%%time` magic, which only times a cell *once*. This way, we can see how long the first pass (compilation) specifically takes.

In [78]:
%%time
# First JAX JIT run (includes compilation cost)
y_jax = f_jax_jit(x_jax).block_until_ready()  # .block_until_ready ensures timing is correct

CPU times: user 95 ms, sys: 18.2 ms, total: 113 ms
Wall time: 59.8 ms


Now that the function is compiled, it should run more quickly. Let's find out!

In [79]:
%%timeit
# Subsequent JAX JIT run (cached, fast!)
y_jax = f_jax_jit(x_jax).block_until_ready()

61.9 ms ± 1.68 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


Note how the second JIT run was indeed faster than the first. This is now faster than both our un-jitted `JAX` code and the original `numpy` code (by a factor of 5!).

Note that, in line with the JAX coding philosophy, the speed increases from jitting a function come with extra constraints. For example, control flow (such is branching `if` statements) can be tricky in jitted functions. The reason for this is that when compiling, `JAX` uses the first function execution to trace through the stack. But each code execution will only move through one of the `if` blocks if the `if` block is conditional on the input, so `jax.jit` won't be aware of what the other code block is doing! For more information on this topic, see https://docs.jax.dev/en/latest/control-flow.html#control-flow.  

## Autodifferentiation in `JAX`
Because `JAX` has the capability to track operations within a Python function, it can also chain-rule those operations together to calculate the derivative of a function. This strategy is known as automatic differentiation, or "autodiff." We can see this play out quite neatly here:

In [64]:
grad_func = grad(jax_function)

Note that this gradient is only defined for scalar-output functions. So, let's pass a scalar:

In [68]:
grad_func(1.0)

Array(0., dtype=float32, weak_type=True)

We have a gradient of 0! Since our function returns 1 for every input value, within machine precision...

In [70]:
y_jax

Array([1.        , 1.        , 1.        , ..., 1.0000001 , 0.99999994,
       1.        ], dtype=float32)

... the gradient should indeed be 0.

You may be wondering why calculating the gradient is worthwhile. It turns out that this is a very helpful quantity to calculate in optimization problems (such as MCMC, or machine learning—for what JAX was developed) and physical systems (such as evolving the orbits of a binary star).

Now we have a handle on the basics of JAX! In the next notebook, we'll explore how to scale this up to multiple cores on TIKE.

# Resources
- [JAX documentation](https://docs.jax.dev/en/latest/index.html)
- [common "gotchas" in JAX](https://docs.jax.dev/en/latest/notebooks/Common_Gotchas_in_JAX.html)
- [Introduction to automatic differentiation](https://www.mathworks.com/help/optim/ug/autodiff-background.html)